## 01 - Data Exploration, Cleaning & Feature Engineering

**Module:** IT5022 Fundamentals of Machine Learning
**Dataset:** NYC Taxi Trip Duration (Kaggle, 2016 NYC TLC records) — https://www.kaggle.com/competitions/nyc-taxi-trip-duration
**Task:** Supervised regression — predict `trip_duration` (seconds) at the moment a trip starts.

In [2]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42          # fixed seed so every run gives the same sample and split
SAMPLE_SIZE = 200_000      # rows used for modelling
TEST_SIZE = 0.20

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../report/figures", exist_ok=True)

### 1. Load the data

In [3]:
DATA_PATH = "../data/raw/train.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["pickup_datetime", "dropoff_datetime"])
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded 1,458,644 rows x 11 columns


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [4]:
# Column types, missing values and duplicate IDs
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
print("Duplicate trip IDs:", df["id"].duplicated().sum())
overview

Duplicate trip IDs: 0


,dtype,missing,unique
id,str,0,1458644
vendor_id,int64,0,2
pickup_datetime,datetime64[us],0,1380222
dropoff_datetime,datetime64[us],0,1380377
passenger_count,int64,0,10
pickup_longitude,float64,0,23047
pickup_latitude,float64,0,45245
dropoff_longitude,float64,0,33821
dropoff_latitude,float64,0,62519
store_and_fwd_flag,str,0,2


### Attribute description

| Column | Meaning | Used as feature? |
|---|---|---|
| `id` | Unique trip identifier | No (identifier) |
| `vendor_id` | Data provider (1 or 2) | Yes |
| `pickup_datetime` | Meter start time | Yes, as hour / weekday / month |
| `dropoff_datetime` | Meter stop time | **No — dropoff minus pickup *is* the target (leakage)** |
| `passenger_count` | Driver-entered passenger count | Yes |
| `pickup_longitude/latitude` | Pickup GPS location | Yes |
| `dropoff_longitude/latitude` | Dropoff GPS location | Yes (destination is known when the trip is booked) |
| `store_and_fwd_flag` | Record held in vehicle memory before sending (Y/N) | Yes (0/1) |
| `trip_duration` | Trip length in seconds | **Target** |